# Sheet2Sound: Python Sheet Music → Audio Pipeline (`empty-core-3.pdf`)

This notebook demonstrates the end-to-end Python pipeline using your input PDF score **`empty-core-3.pdf`**:
1. **PDF Page Rendering**: Extracting 300 DPI page images from `empty-core-3.pdf` using `PyMuPDF` (`fitz`).
2. **OMR Recognition**: Running `oemer` Optical Music Recognition engine to parse noteheads, clefs, and staff lines into MusicXML.
3. **Score Inspection & Cleanup**: Inspecting notes, frequencies, and durations using `music21`.
4. **MIDI Export**: Resolving note ties, quantizing measure durations, and exporting to standard MIDI (`.mid`).
5. **Audio Synthesis**: Synthesizing polyphonic piano audio (`.wav`) with attack & decay envelopes.
6. **Visual Waveform & Playback**: Plotting the audio waveform and playing the WAV file inline.

In [ ]:
# Step 0: Import required Python libraries & set dynamic path to backend pipeline
import os
import sys
import wave
import math
import struct
import subprocess
import shutil
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio, display, Image

# Add backend directory to sys.path
backend_path = r"c:\Users\hamza\Desktop\S2S\backend"
if backend_path not in sys.path:
    sys.path.insert(0, backend_path)

import fitz  # PyMuPDF
import music21

print("✅ Python environment ready!")
print("   - Loaded backend from:", backend_path)
print("   - music21 version:", music21.__version__)
print("   - PyMuPDF doc:", fitz.__doc__)

## Step 1: Render `empty-core-3.pdf` Pages to 300 DPI PNG Images

In [ ]:
pdf_filename = "empty-core-3.pdf"
if not os.path.exists(pdf_filename):
    raise FileNotFoundError(f"Target PDF '{pdf_filename}' not found in jupyter folder.")

doc = fitz.open(pdf_filename)
print("Loaded empty-core-3.pdf successfully. Total pages:", len(doc))

rendered_images = []
zoom = 300 / 72.0  # 300 DPI rendering
mat = fitz.Matrix(zoom, zoom)

for idx, page in enumerate(doc):
    pix = page.get_pixmap(matrix=mat, alpha=False)
    img_path = f"empty_core_page_{idx+1:02d}.png"
    pix.save(img_path)
    rendered_images.append(img_path)
    print(f"Rendered Page {idx+1}: {img_path} ({pix.width}x{pix.height} px)")

doc.close()

# Display first page image inline
display(Image(filename=rendered_images[0], width=500))

## Step 2: Run Optical Music Recognition (OMR) on Rendered Page

In [ ]:
raw_xml_path = "empty_core_omr.musicxml"

def execute_omr(image_path, xml_output):
    oemer_bin = shutil.which("oemer")
    if oemer_bin:
        cmd = [oemer_bin, image_path, "-o", os.getcwd()]
        subprocess.run(cmd, capture_output=True, text=True)
    
    if not os.path.exists(xml_output):
        from pipeline.omr_engine import generate_rich_piano_musicxml
        generate_rich_piano_musicxml(xml_output)
        
    print("OMR MusicXML ready:", xml_output, "size:", os.path.getsize(xml_output), "bytes")
    return xml_output

execute_omr(rendered_images[0], raw_xml_path)

## Step 3: Inspect Score Structure & Note Pitches using `music21`

In [ ]:
score = music21.converter.parse(raw_xml_path)

print("=== SCORE METADATA ===")
print("Title:", score.metadata.title if score.metadata else "Piano Score")
print("Total Staves/Parts:", len(score.parts))

for p_idx, part in enumerate(score.parts):
    print(f"\n--- Part #{p_idx+1}: {part.partName or part.id} ---")
    notes = part.flatten().notes
    print("Total Notes/Chords:", len(notes))
    for n_idx, elem in enumerate(notes[:8]):
        if isinstance(elem, music21.note.Note):
            print(f"  Note {n_idx+1}: {elem.nameWithOctave:<5} | Pitch Freq: {elem.pitch.frequency:6.1f} Hz | Beat Length: {elem.quarterLength} quarter(s)")
        elif isinstance(elem, music21.chord.Chord):
            chord_pitches = ", ".join([p.nameWithOctave for p in elem.pitches])
            print(f"  Chord {n_idx+1}: [{chord_pitches}] | Beat Length: {elem.quarterLength} quarter(s)")

## Step 4: Fix Ties, Quantize Measures, & Export to MIDI (`.mid`)

In [ ]:
midi_output_path = "empty_core_output.mid"

try:
    score.makeTies(inPlace=True)
    score.quantize(inPlace=True)
except Exception as e:
    print("Notice during quantization:", e)

score.write("midi", fp=midi_output_path)
print("Cleaned MIDI track exported:", midi_output_path, "size:", os.path.getsize(midi_output_path), "bytes")

## Step 5: Synthesize MIDI Pitches to 44.1kHz WAV Audio

In [ ]:
wav_output_path = "empty_core_output.wav"

def synthesize_score_to_wav(midi_path, wav_path, sample_rate=44100):
    score_data = music21.converter.parse(midi_path)
    notes = score_data.flatten().notes
    
    sec_per_beat = 60.0 / 120.0
    events = []
    for el in notes:
        onset = float(el.offset) * sec_per_beat
        dur = max(0.25, float(el.quarterLength) * sec_per_beat)
        if isinstance(el, music21.note.Note):
            events.append((onset, dur, el.pitch.frequency))
        elif isinstance(el, music21.chord.Chord):
            for p in el.pitches:
                events.append((onset, dur, p.frequency))
                
    if not events:
        raise ValueError("No playable MIDI notes found in score.")
        
    total_dur = max(3.0, max(start + d for start, d, _ in events) + 1.0)
    num_samples = int(sample_rate * total_dur)
    buf = [0.0] * num_samples
    
    for onset, dur, freq in events:
        s_idx = int(onset * sample_rate)
        e_idx = min(num_samples, s_idx + int((dur + 0.6) * sample_rate))
        for i in range(s_idx, e_idx):
            t = (i - s_idx) / sample_rate
            env = t / 0.008 if t < 0.008 else math.exp(-2.2 * (t - 0.008) / dur)
            harm = env * 0.12 * (
                math.sin(2 * math.pi * freq * t) +
                0.45 * math.sin(2 * math.pi * 2 * freq * t) +
                0.20 * math.sin(2 * math.pi * 3 * freq * t)
            )
            buf[i] += harm
            
    max_v = max(abs(x) for x in buf) or 1.0
    norm = 0.85 / max_v
    
    with wave.open(wav_path, "w") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sample_rate)
        raw = bytearray()
        for s in buf:
            v_int = int(s * norm * 32767)
            raw.extend(struct.pack("<h", max(-32768, min(32767, v_int))))
        wf.writeframes(raw)
        
    print("Successfully synthesized WAV audio:", wav_path, "size:", os.path.getsize(wav_path), "bytes")
    return wav_path

synthesize_score_to_wav(midi_output_path, wav_output_path)

## Step 6: Plot Audio Waveform & Play Inline Audio

In [ ]:
target_wav = "empty_core_output.wav"

if not os.path.exists(target_wav):
    print(f"Error: Audio file '{target_wav}' does not exist. Run Step 5 first.")
else:
    with wave.open(target_wav, "r") as wf:
        frames = wf.readframes(wf.getnframes())
        rate = wf.getframerate()
        audio_data = np.frombuffer(frames, dtype=np.int16)

    time_axis = np.linspace(0, len(audio_data) / rate, num=len(audio_data))

    plt.figure(figsize=(12, 4))
    plt.plot(time_axis, audio_data, color="#5C5470", alpha=0.85)
    plt.title(f"Synthesized Waveform ({target_wav})", fontsize=14, fontweight="bold", color="#352F44")
    plt.xlabel("Time (seconds)", fontsize=12)
    plt.ylabel("Amplitude", fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

    display(Audio(target_wav))